# CX Routing Engine - Native Jupyter Application

This notebook contains the complete CX Routing Engine (Customer Agent, Supervisor Agent, and Developer Copilot) running entirely inside Jupyter cells using your AMD ROCm setup, without needing any external web servers or Ngrok.

In [ ]:
# 1. Install required packages
!pip install -qU vllm langchain-openai langchain langchain-core langchain-community


In [ ]:
import sqlite3
import os

# 2. Setup the Local SQLite Database for Tickets
DB_PATH = "jupyter_cx_routing.db"

def init_db():
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute('''CREATE TABLE IF NOT EXISTS complaints (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        customer_id INTEGER,
        project_name TEXT,
        issue_text TEXT,
        status TEXT,
        assigned_to TEXT,
        priority TEXT,
        eta TEXT,
        category TEXT
    )''')
    c.execute('''CREATE TABLE IF NOT EXISTS timeline (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        complaint_id INTEGER,
        message TEXT,
        timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
        author TEXT
    )''')
    
    # Insert a dummy ticket if empty
    c.execute("SELECT COUNT(*) FROM complaints")
    if c.fetchone()[0] == 0:
        c.execute("INSERT INTO complaints (customer_id, project_name, issue_text, status) VALUES (1, 'BankingApp', 'Database is crashing on login', 'Open')")
    conn.commit()
    conn.close()

init_db()
print("Database initialized!")

In [ ]:
from langchain.tools import tool

# 3. Define the Tools for our Agents

@tool
def check_ticket_status(complaint_id: int) -> str:
    """Fetch the status, priority, and ETA for a given complaint ID."""
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("SELECT status, priority, eta FROM complaints WHERE id=?", (complaint_id,))
    row = c.fetchone()
    conn.close()
    if row:
        return f"Status: {row[0]}, Priority: {row[1]}, ETA: {row[2]}"
    return "Ticket not found."

@tool
def route_complaint(complaint_id: int, specialty_required: str, priority: str, eta: str, category: str, suggested_action: str) -> str:
    """Supervisor tool: Route a complaint to an available engineer based on specialty, set priority and ETA."""
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("UPDATE complaints SET assigned_to=?, priority=?, eta=?, category=?, status='In Progress' WHERE id=?", 
              (specialty_required, priority, eta, category, complaint_id))
    c.execute("INSERT INTO timeline (complaint_id, message, author) VALUES (?, ?, 'Supervisor AI')", 
              (complaint_id, f"Routed to {specialty_required}. Priority: {priority}. ETA: {eta}."))
    conn.commit()
    conn.close()
    return "Complaint routed successfully."

@tool
def update_ticket(complaint_id: int, new_eta: str, developer_message: str) -> str:
    """Developer Copilot tool: Update the ETA of a ticket and leave a message for the customer timeline."""
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("UPDATE complaints SET eta=? WHERE id=?", (new_eta, complaint_id))
    c.execute("INSERT INTO timeline (complaint_id, message, author) VALUES (?, ?, 'Developer')", 
              (complaint_id, developer_message))
    conn.commit()
    conn.close()
    return "Ticket updated successfully."

print("Tools loaded!")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

# 4. Load the AMD ROCm LLM natively (Avoids vLLM Triton driver crashes)
model_id = "Qwen/Qwen2.5-7B-Instruct" # Upgraded to Qwen2.5 for vastly superior tool calling

print(f"Loading tokenizer and model {model_id} onto AMD GPU...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

text_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    max_length=None,
    temperature=0.1,
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=text_pipeline)
chat_model = ChatHuggingFace(llm=llm, model_id=model_id)
print("Qwen Chat Model Ready!")


In [ ]:
# 5. Define our 3 Agents using LangGraph/LangChain
from langchain_core.messages import HumanMessage, SystemMessage

def run_agent(system_prompt, user_query, available_tools=[]):
    """Helper to run our custom agent logic directly in the notebook."""
    # If using tools, bind them
    if available_tools:
        model_with_tools = chat_model.bind_tools(available_tools)
    else:
        model_with_tools = chat_model
        
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_query)
    ]
    
    response = model_with_tools.invoke(messages)
    
    # If the model decided to call a tool, execute it manually (mock agent loop)
    if hasattr(response, 'tool_calls') and response.tool_calls:
        tool_call = response.tool_calls[0]
        print(f"[Agent] Executing tool: {tool_call['name']} with args {tool_call['args']}")
        for t in available_tools:
            if t.name == tool_call['name']:
                tool_result = t.invoke(tool_call['args'])
                print(f"[Tool Result] {tool_result}")
                return tool_result
                
    return response.content

print("Agents Engine Ready!")

In [ ]:
# TEST 1: The Customer Agent
print("--- Testing Customer Agent ---")
customer_prompt = "You are a Customer Agent. The user's Ticket ID is 1. Use your tools to check their ticket status if asked."
reply = run_agent(customer_prompt, "What is the status of my ticket?", available_tools=[check_ticket_status])
print(f"\nCustomer AI Reply: {reply}")

In [ ]:
# TEST 2: The Supervisor Agent
print("\n--- Testing Supervisor Agent ---")
supervisor_prompt = "You are a Supervisor AI. Route ticket ID 1 to a Database Admin, set priority to HIGH, ETA to 4 hours, category Escalated, and suggest they check the DB logs."
reply = run_agent(supervisor_prompt, "Please route ticket 1.", available_tools=[route_complaint])
print(f"\nSupervisor AI Reply: {reply}")

In [ ]:
# TEST 3: The Developer Copilot Agent
print("\n--- Testing Developer Copilot ---")
dev_prompt = "You are a Developer Copilot. The developer wants to update ticket ID 1's ETA to '2 days' and leave the message 'Working on restoring the database cluster.' Use your update_ticket tool."
reply = run_agent(dev_prompt, "Update ticket 1.", available_tools=[update_ticket])
print(f"\nDev Copilot Reply: {reply}")

In [ ]:
# Verify Database Changes
import pandas as pd
print("\n--- Current Database State ---")
conn = sqlite3.connect(DB_PATH)
display(pd.read_sql_query("SELECT * FROM complaints", conn))
print("\n--- Customer Timeline ---")
display(pd.read_sql_query("SELECT * FROM timeline", conn))
conn.close()